In [1]:
import pandas as pd
import numpy as np
from nba_overunder_ml import generate_nba_dataset, train_nba_models, predict_specific_matchup

In [2]:
df = generate_nba_dataset(seasons=['2023-24', '2022-23'])
x = train_nba_models(df)

0        99.28
1       106.32
2       101.92
3       101.04
4        94.44
         ...  
2455    103.68
2456    113.56
2457    105.04
2458    104.04
2459    105.24
Length: 2460, dtype: float64


In [3]:
reg_model, clf_model, features = x['reg_model'], x['clf_model'], x['X_te']
print(features)

      home_pace  away_pace  home_efg_pct  away_efg_pct  home_reb_pct  \
1949     101.24      99.32      0.505635      0.555018      0.538462   
1480     103.36     100.56      0.565451      0.516461      0.476744   
1087      93.68      97.16      0.664539      0.605812      0.542857   
2349     100.04     100.80      0.515148      0.534930      0.540816   
1714      98.16      96.16      0.500432      0.608560      0.469136   
...         ...        ...           ...           ...           ...   
1521     108.08     108.12      0.479000      0.430809      0.613861   
839      100.92      96.68      0.494604      0.664482      0.500000   
1711      99.44      99.44      0.641704      0.499695      0.469136   
511       92.48      95.28      0.608541      0.460157      0.493976   
1952     112.20     110.92      0.499550      0.414689      0.538462   

      away_reb_pct  sportsbook_line  
1949      0.461538            223.0  
1480      0.523256            237.0  
1087      0.457143   

In [4]:
from nba_api.stats.endpoints import leaguedashteamstats
import pandas as pd
import numpy as np

def get_team_stats(season="2024-25", season_type="Regular Season"):
    df = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star=season_type,
        per_mode_detailed="PerGame"
    ).get_data_frames()[0]

    keep = [
        "TEAM_ID", "TEAM_NAME",
        "FGA", "FG3A", "FG3M", "FG_PCT", "FG3_PCT",
        "OREB", "TOV", "FTA", "DREB"
    ]
    df = df[keep].copy()

    df["FG2M"] = (df["FGA"] * df["FG_PCT"]) - (df["FG3A"] * df["FG3_PCT"])
    df["FG2A"] = df["FGA"] - df["FG3A"]
    df["FG2_PCT"] = np.where(df["FG2A"] > 0, df["FG2M"] / df["FG2A"], np.nan)
    df["Pace"] = df["FGA"] - df["OREB"] + df["TOV"] + 0.44 * df["FTA"]

    return df

team_stats = get_team_stats("2024-25")
nuggets_stats = team_stats[team_stats['TEAM_NAME']=='Denver Nuggets'].iloc[0]
celtics_stats = team_stats[team_stats['TEAM_NAME']=='Boston Celtics'].iloc[0]
nuggets_stats['FG_PCT']

np.float64(0.506)

In [5]:
def build_team_dict(team_name, team_stats, opp_stats):
    t =team_stats# team_stats.loc[team_stats["TEAM_NAME"] == team_name].iloc[0]
    o = opp_stats#.loc[opp_stats["TEAM_NAME"] == team_name].iloc[0]

    fg3m = t["FG3M"]
    fga = t["FGA"]
    fgm = t["FG2M"]+ t["FG3M"]
    fgo = o["FG2M"]+ o["FG3M"]
    off_reb_home = t["OREB"]
    def_reb_home = t["DREB"]
    off_reb_away = o["OREB"]
    def_reb_away = o["DREB"]
    act = 2*t["FG2M"]+3*t["FG3M"] +2*o["FG2M"]+3*o["FG3M"] 

    rng = np.random.default_rng(42)

    return {
        "home_pace": float(t["Pace"]),
        "away_pace": float(o["Pace"]),
        "home_efg_pct": float((t["FG2M"] + 0.5 * t["FG3M"]) / t["FGA"]),
        "away_efg_pct": float((o["FG2M"] + 0.5 * o["FG3M"]) / o["FGA"]),
        "home_reb_pct": float((off_reb_home+def_reb_home) / (off_reb_home + def_reb_home+off_reb_away + def_reb_away)),
        "away_reb_pct": float((off_reb_away+def_reb_away) / (off_reb_home + def_reb_home+off_reb_away + def_reb_away)),
        "home_fga": float(t["FGA"]),
        "away_fga": float(o["FGA"]),
        "home_3pa": float(t["FG3A"]),
        "away_3pa": float(o["FG3A"]),
        "home_2p_pct": float(t["FG2_PCT"]),
        "away_2p_pct": float(o["FG2_PCT"]),
        "home_3p_pct": float(t["FG3_PCT"]),
        "away_3p_pct": float(o["FG3_PCT"]),
        "home_oreb": float(t["OREB"]),
        "away_oreb": float(o["OREB"]),
        "home_tov": float(t["TOV"]),
        "away_tov": float(o["TOV"]),
        "home_fta": float(t["FTA"]),
        "away_fta": float(o["FTA"]),   
        "home_fg2_pct": float(t["FG2_PCT"]),
        "away_fg2_pct": float(o["FG2_PCT"]),
        'sportsbook_line' : np.mean(np.round((act + rng.normal(0, 5, len(t))) * 2) / 2)
        
    }   
newceltics_dict = build_team_dict("Boston Celtics", celtics_stats, nuggets_stats)
newnuggets_dict = build_team_dict("Denver Nuggets", nuggets_stats, celtics_stats)
print(newceltics_dict)


{'home_pace': 98.904, 'away_pace': 103.15199999999999, 'home_efg_pct': 0.3638044444444445, 'away_efg_pct': 0.43924721603563477, 'home_reb_pct': 0.49780219780219775, 'away_reb_pct': 0.5021978021978022, 'home_fga': 90.0, 'away_fga': 89.8, 'home_3pa': 48.2, 'away_3pa': 31.9, 'home_2p_pct': 0.5703923444976078, 'away_2p_pct': 0.5776234887737479, 'home_3p_pct': 0.368, 'away_3p_pct': 0.376, 'home_oreb': 11.4, 'away_oreb': 11.2, 'home_tov': 11.9, 'away_tov': 14.3, 'home_fta': 19.1, 'away_fta': 23.3, 'home_fg2_pct': 0.5703923444976078, 'away_fg2_pct': 0.5776234887737479, 'sportsbook_line': np.float64(203.96666666666667)}


In [6]:
FEATURES = [
    'home_efg_pct',
    'away_efg_pct',
    'home_reb_pct',
    'away_reb_pct',
    'home_pace',
    'away_pace',
    'sportsbook_line']
#   'home_fga', 'away_fga',
#    'home_3pa', 'away_3pa',
#    'home_2p_pct', 'away_2p_pct',
#    'home_3p_pct', 'away_3p_pct',
#    'home_oreb', 'away_oreb',
#    'home_tov', 'away_tov',
#    'home_fta', 'away_fta',
#    'home_fg2_pct', 'away_fg2_pct',
def make_model_frame(df):
    return df[FEATURES].copy()

newceltics_dict=make_model_frame(pd.DataFrame([newceltics_dict], columns=FEATURES))
newnuggets_dict=make_model_frame(pd.DataFrame([newnuggets_dict], columns=FEATURES)) 
newceltics_dict

,home_efg_pct,away_efg_pct,home_reb_pct,away_reb_pct,home_pace,away_pace,sportsbook_line
0,0.363804,0.439247,0.497802,0.502198,98.904,103.152,203.966667


In [8]:
context = {
    'team_a_rest': 2,   # Celtics rest days
    'team_b_rest': 0,   # Nuggets rest days
    'is_b2b_a': 0,      # Celtics not on back-to-back
    'is_b2b_b': 1       # Nuggets on back-to-back
}
newceltics_dict
predict_specific_matchup(
    newceltics_dict,
    newnuggets_dict,
    context,
    sportsbook_line=210.5,
    sportsbook_odds_decimal=1.909,
    reg_model=reg_model,
    clf_model=clf_model)

                                     home_pace  \
0  0    98.904
Name: home_pace, dtype: float64   

                                     away_pace  \
0  0    98.904
Name: away_pace, dtype: float64   

                                       home_efg_pct  \
0  0    0.363804
Name: home_efg_pct, dtype: float64   

                                       away_efg_pct  \
0  0    0.363804
Name: away_efg_pct, dtype: float64   

                                       home_reb_pct  \
0  0    0.497802
Name: home_reb_pct, dtype: float64   

                                       away_reb_pct  sportsbook_line  
0  0    0.497802
Name: away_reb_pct, dtype: float64            210.5  
--- MATCHUP PREDICTION ---
Sportsbook Line: 210.5
Model Projected Points: 207.1
Probability of OVER: 35.4%
✅ EDGE FOUND: Bet 6.42% of your bankroll on the UNDER.


(np.float64(207.1232182130865), np.float64(0.3538609511621469), 0)